# Cross-Dataset Validation — RxRx1

Evaluates how well IPP models trained on SLiMIA (3D spheroids) generalise
to **RxRx1** (2D monolayer fluorescence microscopy), testing robustness under
severe geometric and acquisition domain shift.

Models are **transferred directly from SLiMIA without fine-tuning**,
isolating representation robustness rather than adaptation capacity.

| Model | Cross-domain acc (paper) |
|-------|-------------------------|
| ImageShapeFusion | 0.7687 |
| HMTT | 0.7328 |
| CoAtNet-0 | 0.6559 |

**IPP formulation on RxRx1:**  
Predict `experiment` (51 cls), `plate` (4 cls), `site` (2 cls),
`cell_type` (4 cls), and `sirna` (1139 cls) from fluorescence images.

> **Note:** The CoAtNet RxRx1 notebook was unavailable; this notebook
> reconstructs it from the CoAtNet architecture + RxRx1 data pipeline
> following the same protocol as the other two models.

In [ ]:
# !pip install timm opencv-python --quiet

In [ ]:
import os, random, joblib
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import cv2
from PIL import Image
import tifffile
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import timm

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score,
                              recall_score, f1_score)
from sklearn.utils.class_weight import compute_class_weight

In [ ]:
# Config 
class CFG:
    # RxRx1 shape-feature CSV (pre-computed on channel 1 images)
    rxrx1_csv   = "../data/rxrx1/shape_features_channel1.csv"
    output_dir  = "../results/cross_dataset/"
    ckpt_dir    = "../checkpoints/ipp/"

    # SLiMIA label encoders (saved during IPP training)
    encoder_dir = "../results/ipp/"

    # RxRx1 IPP attributes (Table 8 in paper)
    label_columns = ["cell_type", "site", "plate", "experiment"]

    # Shape features extracted from RxRx1 nuclei segmentation
    shape_features = [
        "num_nuclei", "area_mean", "area_std",
        "perimeter_mean", "perimeter_std",
        "eccentricity_mean", "eccentricity_std",
        "solidity_mean", "compactness_mean"
    ]

    # Fusion transformer params (must match SLiMIA training)
    d_model  = 256; n_heads = 4; n_layers = 3
    ff_dim   = 512; dropout = 0.1

    image_size  = 224
    batch_size  = 128
    lr          = 1e-4
    weight_decay= 1e-2
    patience    = 20
    num_epochs  = 100   # shorter — transfer setting
    seed        = 42
    num_workers = 4
    device      = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs(CFG.output_dir, exist_ok=True)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

set_seed(CFG.seed)

In [ ]:
# Load RxRx1 CSV 
df = pd.read_csv(CFG.rxrx1_csv)
df = df[df["channel"] == 1].reset_index(drop=True)   # channel 1 only
df["img_path"] = df["img_path"].astype(str).str.strip()

# Encode sirna_id for stratified splitting
df["sirna_id_enc"] = LabelEncoder().fit_transform(df["sirna_id"].astype(str))

# Encode label columns
label_encoders = {}
for col in CFG.label_columns:
    le = LabelEncoder()
    df[col] = df[col].astype(str)
    df[col + "_enc"] = le.fit_transform(df[col])
    label_encoders[col] = le

label_dims = {col: df[col + "_enc"].nunique() for col in CFG.label_columns}
print("Label dims:", label_dims)
print(f"Total samples: {len(df)}")

# 70 / 15 / 15 split stratified by sirna_id
train_df, temp_df = train_test_split(
    df, test_size=0.30, random_state=CFG.seed,
    stratify=df["sirna_id_enc"])
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=CFG.seed,
    stratify=temp_df["sirna_id_enc"])

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

# Shape normalisation stats
shape_mean = train_df[CFG.shape_features].mean().values.astype(np.float32)
shape_std  = train_df[CFG.shape_features].std().replace(0,1).values.astype(np.float32)

# Class weights
class_weights = {}
for col in CFG.label_columns:
    y  = train_df[col + "_enc"].values
    cls = np.unique(y)
    w  = compute_class_weight("balanced", classes=cls, y=y)
    full_w = np.ones(len(label_encoders[col].classes_), dtype=np.float32)
    for c, wv in zip(cls, w):
        full_w[c] = wv
    class_weights[col] = torch.tensor(full_w).to(CFG.device)

In [ ]:
# Image loader (RxRx1 is grayscale fluorescence) 
def load_rxrx1_image(path: str) -> torch.Tensor:
    """Loads a grayscale fluorescence image → (1, H, W) float tensor in [0,1]."""
    try:
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is None: raise ValueError()
    except Exception:
        arr = tifffile.imread(path)
        if arr.ndim > 2: arr = arr[..., 0]
        img = arr
    img = cv2.resize(img, (CFG.image_size, CFG.image_size))
    return torch.tensor(img, dtype=torch.float32).unsqueeze(0) / 255.0


# Datasets 
class RxRx1Dataset(Dataset):
    """For CoAtNet: returns (rgb_image, labels)."""
    def __init__(self, frame, transform=None):
        self.df        = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        img  = load_rxrx1_image(row["img_path"])
        # Repeat to 3 channels for RGB backbone
        img  = img.repeat(3, 1, 1)
        lbl  = torch.tensor([int(row[c + "_enc"]) for c in CFG.label_columns],
                             dtype=torch.long)
        return img, lbl


class RxRx1FusionDataset(Dataset):
    """For ImageShapeFusion: returns (grayscale_image, shape_vec, labels)."""
    def __init__(self, frame):
        self.df = frame.reset_index(drop=True)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img      = load_rxrx1_image(row["img_path"])   # (1,H,W)
        shape_v  = torch.tensor(
            row[CFG.shape_features].values.astype(np.float32))
        labels   = torch.tensor([int(row[c + "_enc"]) for c in CFG.label_columns],
                                 dtype=torch.long)
        return img, shape_v, labels


# Standard RGB transform for CoAtNet
rgb_tf = T.Compose([
    T.Normalize([0.5]*3, [0.5]*3)   # images already resized + cast in loader
])

train_loader_rgb  = DataLoader(RxRx1Dataset(train_df),
    batch_size=CFG.batch_size, shuffle=True,  num_workers=CFG.num_workers)
val_loader_rgb    = DataLoader(RxRx1Dataset(val_df),
    batch_size=CFG.batch_size, shuffle=False, num_workers=CFG.num_workers)
test_loader_rgb   = DataLoader(RxRx1Dataset(test_df),
    batch_size=CFG.batch_size, shuffle=False, num_workers=CFG.num_workers)

train_loader_fus  = DataLoader(RxRx1FusionDataset(train_df),
    batch_size=CFG.batch_size, shuffle=True,  num_workers=CFG.num_workers)
val_loader_fus    = DataLoader(RxRx1FusionDataset(val_df),
    batch_size=CFG.batch_size, shuffle=False, num_workers=CFG.num_workers)
test_loader_fus   = DataLoader(RxRx1FusionDataset(test_df),
    batch_size=CFG.batch_size, shuffle=False, num_workers=CFG.num_workers)

In [ ]:
# Models 
# All architectures are defined identically to the SLiMIA IPP notebook
# so that weights transfer cleanly.

class MultiTaskBackbone(nn.Module):
    """Generic multi-head backbone (CoAtNet, ViT, ConvNeXt)."""
    def __init__(self, backbone_name, label_dims, pretrained=False, in_chans=3):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained,
                                          num_classes=0, in_chans=in_chans)
        D = self.backbone.num_features
        self.heads = nn.ModuleDict({
            lab: nn.Linear(D, dim) for lab, dim in label_dims.items()
        })

    def forward(self, x):
        feat = self.backbone(x)
        return {lab: head(feat) for lab, head in self.heads.items()}


class ImageShapeFusionTransformer(nn.Module):
    """ConvNeXt-Tiny + shape token fusion (RxRx1 uses 1-channel input)."""
    def __init__(self, label_dims, shape_mean, shape_std):
        super().__init__()
        n_shape = len(CFG.shape_features)
        D       = CFG.d_model
        # in_chans=1 for grayscale fluorescence
        self.backbone   = timm.create_model("convnext_tiny", pretrained=True,
                                            num_classes=0, in_chans=1)
        self.image_proj = nn.Linear(self.backbone.num_features, D)
        self.shape_proj = nn.ModuleList([nn.Linear(1, D) for _ in range(n_shape)])
        self.n_shape    = n_shape
        self.pos_embed  = nn.Parameter(torch.randn(1, 1 + n_shape, D) * 0.02)
        enc_layer       = nn.TransformerEncoderLayer(
            d_model=D, nhead=CFG.n_heads, dim_feedforward=CFG.ff_dim,
            dropout=CFG.dropout, activation="gelu")
        self.transformer = nn.TransformerEncoder(enc_layer, CFG.n_layers)
        self.norm        = nn.LayerNorm(D)
        self.heads       = nn.ModuleDict({
            lab: nn.Sequential(
                nn.LayerNorm(D), nn.Linear(D, D), nn.GELU(),
                nn.Dropout(0.2), nn.Linear(D, ncls)
            ) for lab, ncls in label_dims.items()
        })
        self.register_buffer("shape_mean", torch.tensor(shape_mean))
        self.register_buffer("shape_std",  torch.tensor(shape_std))

    def forward(self, images, shape_feats):
        img_tok  = self.image_proj(self.backbone(images)).unsqueeze(1)
        shape_n  = (shape_feats - self.shape_mean) / (self.shape_std + 1e-6)
        s_toks   = torch.cat([self.shape_proj[i](shape_n[:, i:i+1]).unsqueeze(1)
                              for i in range(self.n_shape)], dim=1)
        seq      = torch.cat([img_tok, s_toks], dim=1) + self.pos_embed
        fused    = self.norm(self.transformer(seq.permute(1,0,2))[0])
        return {lab: head(fused) for lab, head in self.heads.items()}


class HMTT(nn.Module):
    """Hierarchical Multi-Task Transformer (ViT-B/16 encoder)."""
    def __init__(self, label_dims, label_order):
        super().__init__()
        self.label_order = label_order
        self.all_labels  = list(label_dims.keys())
        self.encoder     = timm.create_model("vit_base_patch16_224",
                                             pretrained=True, num_classes=0)
        D = self.encoder.num_features
        self.embeds = nn.ModuleDict({
            lab: nn.Embedding(label_dims[lab], D) for lab in self.all_labels
        })
        self.heads = nn.ModuleDict({
            lab: nn.Sequential(
                nn.LayerNorm(D*2), nn.Linear(D*2, D),
                nn.GELU(), nn.Dropout(0.2), nn.Linear(D, label_dims[lab])
            ) for lab in self.all_labels
        })

    def forward(self, x, labels=None, teacher_forcing=True):
        B, feat = x.size(0), self.encoder(x)
        zeros   = torch.zeros(B, feat.size(1), device=x.device)
        outputs, chosen = {}, {}
        for i, lab in enumerate(self.label_order):
            ctx = zeros.clone()
            for j in range(i):
                prev = self.label_order[j]
                idx  = (labels[:, self.all_labels.index(prev)]
                        if teacher_forcing and labels is not None
                        else chosen[prev])
                ctx  = ctx + self.embeds[prev](idx)
            logits       = self.heads[lab](torch.cat([feat, ctx], dim=1))
            outputs[lab] = logits
            chosen[lab]  = logits.argmax(1)
        for lab in self.all_labels:
            if lab not in self.label_order:
                outputs[lab] = self.heads[lab](torch.cat([feat, zeros], dim=1))
        return outputs

HMTT_ORDER = [l for l in
    ["cell_type", "experiment", "plate", "site"]
    if l in CFG.label_columns]

In [ ]:
# Training & Evaluation helpers 
def eval_model(model, loader, is_fusion=False, is_hmtt=False):
    model.eval()
    preds, tgts = defaultdict(list), defaultdict(list)
    with torch.no_grad():
        for batch in tqdm(loader, desc="Eval", leave=False):
            if is_fusion:
                imgs, shp, lbl = batch
                imgs = imgs.to(CFG.device); shp = shp.to(CFG.device)
                lbl  = lbl.to(CFG.device)
                outs = model(imgs, shp)
            else:
                imgs, lbl = batch
                imgs = imgs.to(CFG.device); lbl = lbl.to(CFG.device)
                outs = model(imgs, labels=None, teacher_forcing=False) \
                       if is_hmtt else model(imgs)
            for i, c in enumerate(CFG.label_columns):
                preds[c] += outs[c].argmax(1).cpu().tolist()
                tgts[c]  += lbl[:, i].cpu().tolist()

    results = {}
    for c in CFG.label_columns:
        t, p = np.array(tgts[c]), np.array(preds[c])
        results[c] = dict(
            acc  = accuracy_score(t, p),
            prec = precision_score(t, p, average="macro", zero_division=0),
            rec  = recall_score(t, p, average="macro", zero_division=0),
            f1   = f1_score(t, p, average="macro", zero_division=0),
        )
    macro = {k: np.mean([results[c][k] for c in CFG.label_columns])
             for k in ["acc","prec","rec","f1"]}
    return {"per_label": results, "macro": macro}


def train_rxrx1(model_name, model, train_loader, val_loader,
                is_fusion=False, is_hmtt=False):
    model = model.to(CFG.device)
    criterions = {c: nn.CrossEntropyLoss(weight=class_weights[c])
                  for c in CFG.label_columns}
    opt = torch.optim.AdamW(model.parameters(),
                            lr=CFG.lr, weight_decay=CFG.weight_decay)
    sched  = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", patience=5)
    scaler = torch.amp.GradScaler(enabled=(CFG.device == "cuda"))
    best, patience_cnt = 0.0, 0
    ckpt = os.path.join(CFG.output_dir, f"{model_name}_rxrx1_best.pth")

    for epoch in range(1, CFG.num_epochs + 1):
        model.train()
        for batch in tqdm(train_loader, desc=f"{model_name} ep{epoch}", leave=False):
            if is_fusion:
                imgs, shp, lbl = batch
                imgs = imgs.to(CFG.device); shp = shp.to(CFG.device)
                lbl  = lbl.to(CFG.device)
            else:
                imgs, lbl = batch
                imgs = imgs.to(CFG.device); lbl = lbl.to(CFG.device)
                shp = None

            opt.zero_grad()
            with torch.amp.autocast(device_type=CFG.device):
                if is_hmtt:
                    outs = model(imgs, labels=lbl, teacher_forcing=True)
                elif is_fusion:
                    outs = model(imgs, shp)
                else:
                    outs = model(imgs)
                loss = sum(criterions[c](outs[c], lbl[:, i])
                           for i, c in enumerate(CFG.label_columns))
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()

        val_m = eval_model(model, val_loader, is_fusion, is_hmtt)
        va_acc = val_m["macro"]["acc"]
        sched.step(va_acc)
        print(f"  Epoch {epoch:03d} | Val Acc {va_acc:.4f} | F1 {val_m['macro']['f1']:.4f}")

        if va_acc > best:
            best = va_acc; patience_cnt = 0
            torch.save(model.state_dict(), ckpt)
        else:
            patience_cnt += 1
            if patience_cnt >= CFG.patience:
                print("  Early stopping."); break

    return ckpt

In [ ]:
# Run all three models
# Models are trained FROM SCRATCH on RxRx1 here.
# To reproduce the paper's zero-shot transfer numbers,
# load SLiMIA weights and skip training (see cell below).

cross_results = {}

# 1. ImageShapeFusionTransformer
print("\n" + "="*60 + "\n  ImageShapeFusionTransformer on RxRx1")
fus_model = ImageShapeFusionTransformer(label_dims, shape_mean, shape_std)
ckpt = train_rxrx1("FusionTransformer", fus_model,
                   train_loader_fus, val_loader_fus, is_fusion=True)
fus_model.load_state_dict(torch.load(ckpt, map_location=CFG.device))
test_m = eval_model(fus_model.to(CFG.device), test_loader_fus, is_fusion=True)
cross_results["ImageShapeFusion"] = test_m["macro"]
print("Test:", test_m["macro"])

# 2. HMTT
print("\n" + "="*60 + "\n  HMTT on RxRx1")
hmtt_model = HMTT(label_dims, HMTT_ORDER)
ckpt = train_rxrx1("HMTT", hmtt_model,
                   train_loader_rgb, val_loader_rgb, is_hmtt=True)
hmtt_model.load_state_dict(torch.load(ckpt, map_location=CFG.device))
test_m = eval_model(hmtt_model.to(CFG.device), test_loader_rgb, is_hmtt=True)
cross_results["HMTT"] = test_m["macro"]
print("Test:", test_m["macro"])

# 3. CoAtNet-0 (reconstructed — original notebook unavailable)
print("\n" + "="*60 + "\n  CoAtNet-0 on RxRx1")
coatnet_model = MultiTaskBackbone("coatnet_0_224", label_dims, pretrained=False)
ckpt = train_rxrx1("CoAtNet", coatnet_model,
                   train_loader_rgb, val_loader_rgb)
coatnet_model.load_state_dict(torch.load(ckpt, map_location=CFG.device))
test_m = eval_model(coatnet_model.to(CFG.device), test_loader_rgb)
cross_results["CoAtNet-0"] = test_m["macro"]
print("Test:", test_m["macro"])

In [ ]:
# Zero-shot transfer (paper protocol)
# Load SLiMIA checkpoints directly and evaluate on RxRx1 without fine-tuning.
# This requires that label dims match — they won't unless you retrain the heads.
# For a fair zero-shot comparison, replace the heads with new random heads
# matching RxRx1 label_dims and only transfer the backbone weights.

print("\n─── Zero-shot backbone transfer (backbone only) ───")
slimia_ckpt = os.path.join(CFG.ckpt_dir, "CoAtNet-0_seed42.pth")
if os.path.exists(slimia_ckpt):
    coatnet_transfer = MultiTaskBackbone("coatnet_0_224", label_dims, pretrained=False)
    # Load only backbone weights (heads have different dims)
    slimia_state = torch.load(slimia_ckpt, map_location=CFG.device)
    backbone_state = {k: v for k, v in slimia_state.items()
                      if k.startswith("backbone.")}
    coatnet_transfer.load_state_dict(backbone_state, strict=False)
    # Evaluate with random heads (zero-shot feature quality check)
    test_m = eval_model(coatnet_transfer.to(CFG.device), test_loader_rgb)
    print("Zero-shot (backbone only):", test_m["macro"])
else:
    print(f"SLiMIA checkpoint not found at {slimia_ckpt}")

In [ ]:
# Summary table
rows = []
for name, m in cross_results.items():
    rows.append({"Model": name, **{k: f"{v:.4f}" for k, v in m.items()}})

results_df = pd.DataFrame(rows).set_index("Model")
print("\nCross-Dataset RxRx1 Results (Table 8)")
print(results_df.to_string())

results_df.to_csv(os.path.join(CFG.output_dir, "rxrx1_cross_dataset_results.csv"))
print("Saved.")